# 🔗 Day 149 — Streamlit ↔ FastAPI Full-Stack Integration
## Month 8 | Streamlit + FastAPI Deployment | Deepanshu Garg

---

| Field | Details |
|---|---|
| **Month** | 8 — Streamlit + FastAPI Deployment |
| **Day** | 149 (Month 8, Week 3, Day 4) |
| **Topic** | Streamlit frontend calling FastAPI backend · `requests` library · API key auth in UI · Error handling · Live dashboard from API |
| **Dataset** | FreelanceHub India (500 rows, seed=141) |
| **Deliverables** | `day149_app/streamlit_app.py` · `day149_api/main.py` (extended from Day 148) · `run_stack.py` |
| **Max Score** | 80 pts + 10★ Bonus |
| **GitHub Repo** | Month8-Streamlit-FastAPI-Portfolio |

---

## 🗺️ Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ ✅ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ ✅ |
| 143 | File Upload + Dynamic EDA | 80/80 + 10★ ✅ |
| 144 | ML Model Deployment (Basic) | 80/80 + 10★ ✅ |
| 145 | Advanced Caching + Multi-Page Apps | 80/80 + 10★ ✅ |
| 146 | ML Explainability Dashboard (SHAP) | 80/80 + 10★ ✅ |
| 147 | FastAPI Fundamentals | 80/80 + 10★ ✅ |
| 148 | FastAPI + DB + Auth | 80/80 + 10★ ✅ |
| **149** | **Streamlit ↔ FastAPI Full-Stack** | **← Today** |
| 150 | Docker Basics + Containerization | Upcoming |

---

## 🏗️ What Full-Stack Means Today

```
                ┌─────────────────────────────────────────┐
                │          BROWSER (User sees this)        │
                │                                          │
                │   ┌──────────────────────────────────┐   │
                │   │     Streamlit Frontend           │   │
                │   │   (day149_app/streamlit_app.py)  │   │
                │   │                                  │   │
                │   │  KPI Cards | Charts | Predict    │   │
                │   │  Filter UI | Error messages      │   │
                │   └──────────┬───────────────────────┘   │
                │              │  HTTP requests             │
                │              │  (requests.get/post)       │
                │              ▼                            │
                │   ┌──────────────────────────────────┐   │
                │   │     FastAPI Backend              │   │
                │   │   (day149_api/main.py)           │   │
                │   │                                  │   │
                │   │  /projects/stats  → JSON         │   │
                │   │  /projects/filter → JSON list    │   │
                │   │  /predict         → prediction   │   │
                │   └──────────┬───────────────────────┘   │
                │              │                            │
                │              ▼                            │
                │   ┌──────────────────────────────────┐   │
                │   │     SQLite + Trained Model       │   │
                │   └──────────────────────────────────┘   │
                └─────────────────────────────────────────┘

Ports:  FastAPI  → localhost:8000
        Streamlit → localhost:8501
```

> **Freelance angle:** Every client wants a *dashboard*, not a raw API.  
> Day 149 is the bridge: Streamlit becomes the client-facing UI; FastAPI handles  
> the data and ML logic. This is the architecture behind real BI tools and SaaS dashboards.


---
## 📂 Section 1 — Raw Data
> ⚠️ **NEVER MODIFY THIS SECTION.** Reference only.


In [ ]:
import numpy as np
import pandas as pd

# ── FreelanceHub India — seed=141, 500 rows ──────────────────────────
np.random.seed(141)
n = 500
platforms     = np.random.choice(['Upwork','Freelancer','Fiverr','Toptal'], n, p=[0.35,0.30,0.25,0.10])
skills        = np.random.choice(['Python','SQL','ML','Tableau','Excel','NLP'], n, p=[0.25,0.20,0.20,0.15,0.10,0.10])
experience    = np.random.choice(['Junior','Mid','Senior','Expert'], n, p=[0.25,0.35,0.25,0.15])
project_type  = np.random.choice(['Fixed','Hourly'], n, p=[0.55,0.45])
hours_billed  = np.random.randint(5, 201, n)
hourly_rate   = np.round(np.random.uniform(10, 100, n), 2)
client_rating = np.round(np.random.uniform(2.5, 5.0, n), 1)
status        = np.random.choice(['Completed','In Progress','Cancelled'], n, p=[0.60,0.25,0.15])
project_value = np.round(hours_billed * hourly_rate, 2)
high_value    = (project_value > 5000).astype(int)

RAW_DF = pd.DataFrame({
    'project_id'   : range(1, n+1),
    'platform'     : platforms,
    'skill'        : skills,
    'experience'   : experience,
    'project_type' : project_type,
    'hours_billed' : hours_billed,
    'hourly_rate'  : hourly_rate,
    'client_rating': client_rating,
    'status'       : status,
    'project_value': project_value,
    'high_value'   : high_value
})

print(f"Shape: {RAW_DF.shape}")
print(RAW_DF.head(3).to_string())


---
## 📖 Section 2 — Concept Notes

### 2.1 The `requests` Library — HTTP from Python

`requests` is Python's standard HTTP client. In Streamlit, you use it to call your FastAPI:

```python
import requests

# GET request with API key header
response = requests.get(
    "http://localhost:8000/projects/stats",
    headers={"X-API-Key": "freelancehub-2024-secret"},
    timeout=10   # ← always set a timeout; never hang forever
)

# Check success
if response.status_code == 200:
    data = response.json()   # dict
else:
    st.error(f"API error {response.status_code}: {response.text}")
```

| Method | FastAPI counterpart | Use case |
|---|---|---|
| `requests.get(url, params={...})` | `@app.get(...)` | Fetch data, filter, stats |
| `requests.post(url, json={...})` | `@app.post(...)` | Send prediction input |
| `requests.delete(url)` | `@app.delete(...)` | Remove a record |

### 2.2 Error Handling Pattern — Never Let Streamlit Crash

Always wrap API calls in try/except:

```python
import requests

def safe_api_call(url, headers, params=None, method="get", payload=None):
    """
    Returns (data_dict_or_list, error_message_or_None).
    Streamlit pages check the error before rendering.
    """
    try:
        if method == "get":
            r = requests.get(url, headers=headers, params=params, timeout=10)
        else:
            r = requests.post(url, headers=headers, json=payload, timeout=10)

        if r.status_code == 200:
            return r.json(), None
        elif r.status_code == 401:
            return None, "❌ Unauthorized — check your API key."
        elif r.status_code == 422:
            return None, f"❌ Validation error — check your input fields."
        else:
            return None, f"❌ API returned {r.status_code}: {r.text}"

    except requests.exceptions.ConnectionError:
        return None, "❌ Cannot reach API. Is FastAPI running on port 8000?"
    except requests.exceptions.Timeout:
        return None, "❌ Request timed out (>10s). Try again."
    except Exception as e:
        return None, f"❌ Unexpected error: {e}"
```

### 2.3 API Key Management in Streamlit — Two Approaches

**Dev / local (use today):** Hardcode in a config dict or `st.sidebar` input
```python
API_KEY = "freelancehub-2024-secret"   # fine for local dev
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "http://localhost:8000"
```

**Production / cloud (Day 152 onwards):** Use `st.secrets` — stored in `.streamlit/secrets.toml`:
```toml
[api]
key = "freelancehub-2024-secret"
base_url = "https://your-fastapi-app.onrender.com"
```
```python
API_KEY  = st.secrets["api"]["key"]
BASE_URL = st.secrets["api"]["base_url"]
```

### 2.4 Running Two Servers Together

FastAPI and Streamlit run as separate processes on different ports:

```
Terminal 1:   uvicorn day149_api.main:app --reload --port 8000
Terminal 2:   streamlit run day149_app/streamlit_app.py
```

Or via `run_stack.py` which launches both using `subprocess`:

```python
import subprocess, sys, time

api = subprocess.Popen(
    ["uvicorn", "day149_api.main:app", "--reload", "--port", "8000"]
)
time.sleep(2)   # give FastAPI time to start

st = subprocess.Popen(
    ["streamlit", "run", "day149_app/streamlit_app.py"]
)

try:
    api.wait()
    st.wait()
except KeyboardInterrupt:
    api.terminate()
    st.terminate()
```

### 2.5 `st.cache_data` on API calls — avoid hammering the server

```python
@st.cache_data(ttl=60)   # cache for 60 seconds
def fetch_stats(api_key: str):
    data, err = safe_api_call(f"{BASE_URL}/projects/stats",
                               headers={"X-API-Key": api_key})
    return data, err
```

`ttl=60` means the API call fires once per minute max — critical when deploying
to cloud where you may be charged per call or have rate limits.


---
## 🛠️ Section 3 — Practice Guide (80 pts)

### Architecture Overview

```
day149_app/
    streamlit_app.py      ← you build this (Tasks 1–5)

day149_api/               ← extend from Day 148
    __init__.py
    main.py               ← add /projects/platform-summary endpoint (Task 0)
    database.py
    models.py
    auth.py

run_stack.py              ← Task 6 (bonus)
```

---

### Task 0 — Extend FastAPI: Add `/projects/platform-summary` (10 pts)

In `day149_api/main.py`, add one new endpoint:

```python
@app.get("/projects/platform-summary")
def platform_summary(
    db: Session = Depends(get_db),
    role: str   = Depends(verify_api_key)
):
    """
    Returns per-platform stats:
    [
      {"platform": "Upwork",     "count": 180, "avg_hourly_rate": 54.16, "avg_project_value": 5412.34},
      {"platform": "Fiverr",     "count": 136, "avg_hourly_rate": 54.40, "avg_project_value": 5211.22},
      ...
    ]
    """
    rows = (
        db.query(
            models.Project.platform,
            func.count(models.Project.id).label("count"),
            func.round(func.avg(models.Project.hourly_rate), 2).label("avg_hourly_rate"),
            func.round(func.avg(models.Project.project_value), 2).label("avg_project_value"),
        )
        .group_by(models.Project.platform)
        .order_by(func.count(models.Project.id).desc())
        .all()
    )
    return [
        {
            "platform"          : r.platform,
            "count"             : r.count,
            "avg_hourly_rate"   : float(r.avg_hourly_rate),
            "avg_project_value" : float(r.avg_project_value),
        }
        for r in rows
    ]
```

**Verify (Swagger UI at `http://localhost:8000/docs`):**  
`GET /projects/platform-summary` → returns list of 4 dicts (Upwork, Freelancer, Fiverr, Toptal).

---

### Task 1 — `safe_api_call()` helper + config (10 pts)

Create `day149_app/streamlit_app.py`. Start with:

```python
import streamlit as st
import requests
import pandas as pd
import plotly.express as px

# ── CONFIG ────────────────────────────────────────────────────────────
BASE_URL = "http://localhost:8000"
API_KEY  = "freelancehub-2024-secret"
HEADERS  = {"X-API-Key": API_KEY}

# ── HELPER ────────────────────────────────────────────────────────────
def safe_api_call(url, params=None, method="get", payload=None):
    """
    Returns (data, error).
    data  → dict or list on success, None on failure.
    error → None on success, str message on failure.
    """
    try:
        if method == "get":
            r = requests.get(url, headers=HEADERS, params=params, timeout=10)
        else:
            r = requests.post(url, headers=HEADERS, json=payload, timeout=10)

        if r.status_code == 200:
            return r.json(), None
        elif r.status_code == 401:
            return None, "❌ Unauthorized — check your API key."
        elif r.status_code == 422:
            detail = r.json().get("detail", r.text)
            return None, f"❌ Validation error: {detail}"
        else:
            return None, f"❌ API error {r.status_code}: {r.text}"

    except requests.exceptions.ConnectionError:
        return None, "❌ Cannot reach API. Is FastAPI running on port 8000?"
    except requests.exceptions.Timeout:
        return None, "❌ Request timed out. Try again."
    except Exception as e:
        return None, f"❌ Unexpected: {e}"
```

**Scoring check:**  
- [ ] `BASE_URL`, `API_KEY`, `HEADERS` defined at module level (not inside functions)  
- [ ] `safe_api_call` handles: 200, 401, 422, ConnectionError, Timeout  
- [ ] Returns `(data, error)` tuple — not just data

---

### Task 2 — Page 1: KPI Dashboard (20 pts)

Build the main dashboard page using data from `GET /projects/stats`.

```python
st.set_page_config(page_title="FreelanceHub Dashboard", layout="wide", page_icon="📊")
st.title("📊 FreelanceHub India — Live Dashboard")
st.caption("Data served by FastAPI · Streamlit frontend · FreelanceHub India 500 projects")

# ── KPI CARDS ─────────────────────────────────────────────────────────
@st.cache_data(ttl=60)
def get_stats():
    return safe_api_call(f"{BASE_URL}/projects/stats")

data, err = get_stats()

if err:
    st.error(err)
    st.info("💡 Start FastAPI first:  uvicorn day149_api.main:app --reload --port 8000")
    st.stop()

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("Total Projects",   data["total_projects"])
with col2:
    st.metric("Completed",        data["completed"],
              delta=f"{data['completed']/data['total_projects']*100:.1f}% completion rate")
with col3:
    st.metric("Avg Hourly Rate",  f"${data['avg_hourly_rate']}")
with col4:
    st.metric("High-Value (>$5k)", data["high_value_count"])
```

Then add a **Platform Summary Chart** using data from `GET /projects/platform-summary`:

```python
st.subheader("📈 Platform Performance")

@st.cache_data(ttl=60)
def get_platform_summary():
    return safe_api_call(f"{BASE_URL}/projects/platform-summary")

pdata, perr = get_platform_summary()

if perr:
    st.warning(f"Platform data unavailable: {perr}")
else:
    pf_df = pd.DataFrame(pdata)

    col_a, col_b = st.columns(2)
    with col_a:
        fig1 = px.bar(
            pf_df, x="platform", y="count",
            title=f"Projects by Platform — Upwork leads with {pf_df.iloc[0]['count']} projects",
            color="platform",
            color_discrete_sequence=px.colors.qualitative.Set2,
            text="count"
        )
        fig1.update_layout(showlegend=False)
        st.plotly_chart(fig1, use_container_width=True)

    with col_b:
        fig2 = px.bar(
            pf_df, x="platform", y="avg_project_value",
            title=f"Avg Project Value by Platform — Toptal leads at ${pf_df.sort_values('avg_project_value',ascending=False).iloc[0]['avg_project_value']:,.2f}",
            color="platform",
            color_discrete_sequence=px.colors.qualitative.Pastel,
            text="avg_project_value"
        )
        fig2.update_traces(texttemplate='$%{text:,.0f}')
        fig2.update_layout(showlegend=False)
        st.plotly_chart(fig2, use_container_width=True)
```

**Scoring check:**
- [ ] `st.metric` shows correct values from API response (not hardcoded)
- [ ] Completion rate delta computed dynamically: `completed / total_projects * 100`
- [ ] Chart titles embed actual numbers from API data (NRA-style)
- [ ] `@st.cache_data(ttl=60)` applied to both API calls
- [ ] `st.stop()` called if stats API fails — page doesn't render partial data

---

### Task 3 — Page 2: Project Filter (15 pts)

Add a tabbed interface below the charts:

```python
st.divider()
tab1, tab2 = st.tabs(["🔍 Filter Projects", "🤖 Predict High-Value"])

with tab1:
    st.subheader("🔍 Filter Projects by Status")

    status_choice = st.selectbox(
        "Select project status:",
        ["Completed", "In Progress", "Cancelled"]
    )

    if st.button("🔎 Fetch Projects", key="filter_btn"):
        fdata, ferr = safe_api_call(
            f"{BASE_URL}/projects/filter",
            params={"status": status_choice}
        )

        if ferr:
            st.error(ferr)
        else:
            st.success(f"✅ Found {len(fdata)} projects with status: **{status_choice}**")

            fdf = pd.DataFrame(fdata)

            # Summary row
            col_x, col_y, col_z = st.columns(3)
            with col_x:
                st.metric("Count", len(fdf))
            with col_y:
                st.metric("Avg Hourly Rate", f"${fdf['hourly_rate'].mean():.2f}")
            with col_z:
                st.metric("Avg Project Value", f"${fdf['project_value'].mean():,.2f}")

            # Data table — show key columns only
            st.dataframe(
                fdf[['project_id','platform','skill','experience',
                      'hourly_rate','project_value','client_rating']],
                use_container_width=True,
                hide_index=True
            )

            # NRA Insight box
            top_skill = fdf.groupby('skill')['project_value'].mean().idxmax()
            top_val   = fdf.groupby('skill')['project_value'].mean().max()
            st.info(
                f"💡 **NRA Insight:** Among {status_choice} projects, "
                f"**{top_skill}** generates the highest avg project value at "
                f"**${top_val:,.2f}**. "
                f"Action: prioritise {top_skill} proposals to maximise {status_choice} earnings."
            )
```

**Scoring check:**
- [ ] `params={"status": status_choice}` passed correctly to `safe_api_call`
- [ ] 3 metric cards computed from the *returned* DataFrame (not hardcoded)
- [ ] NRA insight uses `idxmax()` + `max()` from grouped data — no hardcoded strings
- [ ] `st.dataframe` shows only relevant columns (not all 11)

---

### Task 4 — Page 3: Predict High-Value (15 pts)

```python
with tab2:
    st.subheader("🤖 Predict: High-Value Project?")
    st.caption("Sends input to FastAPI `/predict` endpoint · Returns probability + label")

    with st.form("predict_form"):
        c1, c2, c3 = st.columns(3)
        with c1:
            p_platform  = st.selectbox("Platform",   ["Upwork","Freelancer","Fiverr","Toptal"])
            p_skill     = st.selectbox("Skill",       ["Python","SQL","ML","Tableau","Excel","NLP"])
        with c2:
            p_exp       = st.selectbox("Experience",  ["Junior","Mid","Senior","Expert"])
            p_proj_type = st.selectbox("Project Type",["Fixed","Hourly"])
        with c3:
            p_hours     = st.slider("Hours Billed", 5, 200, 80)
            p_rate      = st.slider("Hourly Rate ($)", 10.0, 100.0, 55.0, step=0.5)
            p_rating    = st.slider("Client Rating", 2.5, 5.0, 4.2, step=0.1)

        submitted = st.form_submit_button("🚀 Predict")

    if submitted:
        payload = {
            "platform"    : p_platform,
            "skill"       : p_skill,
            "experience"  : p_exp,
            "project_type": p_proj_type,
            "hours_billed": p_hours,
            "hourly_rate" : p_rate,
            "client_rating": p_rating
        }

        pred_data, pred_err = safe_api_call(
            f"{BASE_URL}/predict",
            method="post",
            payload=payload
        )

        if pred_err:
            st.error(pred_err)
        else:
            prob    = pred_data["probability"]
            label   = pred_data["prediction"]
            proj_v  = p_hours * p_rate   # computed locally for display

            if label == 1:
                st.success(f"✅ **HIGH-VALUE project** — Probability: {prob:.1%}")
                st.balloons()
            else:
                st.warning(f"⚠️ **NOT high-value** — Probability: {prob:.1%}")

            # Display input summary
            st.write("**Input summary:**")
            summary_df = pd.DataFrame([payload])
            summary_df["estimated_value"] = f"${proj_v:,.2f}"
            st.dataframe(summary_df, use_container_width=True, hide_index=True)

            # Business interpretation box
            st.info(
                f"💡 **NRA Insight:** Estimated project value = **${proj_v:,.2f}** "
                f"({p_hours}h × ${p_rate}/hr). "
                f"Model confidence: **{prob:.1%}**. "
                f"{'Proceed — strong high-value signal.' if label==1 else 'Reconsider scope: increase hours or hourly rate to cross the $5,000 threshold.'}"
            )
```

**Scoring check:**
- [ ] Form uses `st.form` + `st.form_submit_button` (not bare button)
- [ ] `payload` dict keys match Day 147/148 `ProjectInput` Pydantic schema exactly
- [ ] `st.success` / `st.warning` chosen conditionally on label, not hardcoded
- [ ] `prob` displayed as percentage: `f"{prob:.1%}"`
- [ ] NRA insight references computed `proj_v` — not a static string

---

### Task 5 — Sidebar: API Health Check + Docs Link (10 pts)

```python
# ── SIDEBAR ───────────────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ API Settings")

    # Live API health check
    health_data, health_err = safe_api_call(f"{BASE_URL}/")

    if health_err:
        st.error("🔴 API Offline")
        st.code("uvicorn day149_api.main:app --reload --port 8000", language="bash")
    else:
        st.success("🟢 API Online")
        st.json(health_data)   # shows version, endpoints list

    st.divider()
    st.markdown(f"[📚 API Docs (Swagger)]({BASE_URL}/docs)")
    st.markdown(f"[📄 ReDoc]({BASE_URL}/redoc)")

    st.divider()
    st.write("**API Key in use:**")
    st.code(API_KEY[:20] + "...", language="text")

    # Manual cache clear
    if st.button("🔄 Clear Cache"):
        st.cache_data.clear()
        st.success("Cache cleared!")
        st.rerun()

    st.divider()
    st.caption("FreelanceHub India · 500 projects · seed=141")
    st.caption("FastAPI port: 8000 · Streamlit port: 8501")
```

**Scoring check:**
- [ ] API health check calls `GET /` — shows "🟢 API Online" or "🔴 API Offline"
- [ ] When offline, shows the `uvicorn` command to start it (not just an error)
- [ ] Swagger + ReDoc links rendered as clickable markdown links
- [ ] Cache clear button calls `st.cache_data.clear()` + `st.rerun()`

---

### Task 6 ★ BONUS — `run_stack.py` (10 pts)

Create `run_stack.py` at project root:

```python
"""
run_stack.py — launch FastAPI + Streamlit together.
Usage: python run_stack.py
       Ctrl+C to stop both servers.
"""
import subprocess
import sys
import time
import signal

print("🚀 Starting FreelanceHub full stack...")
print("   FastAPI  → http://localhost:8000")
print("   Swagger  → http://localhost:8000/docs")
print("   Streamlit→ http://localhost:8501")
print("   Press Ctrl+C to stop both servers.")
print()

api_proc = subprocess.Popen(
    ["uvicorn", "day149_api.main:app", "--reload", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("⏳ Waiting for FastAPI to initialise (2s)...")
time.sleep(2)

if api_proc.poll() is not None:
    print("❌ FastAPI failed to start. Check day149_api/main.py for errors.")
    sys.exit(1)

print("✅ FastAPI running.")

st_proc = subprocess.Popen(
    ["streamlit", "run", "day149_app/streamlit_app.py",
     "--server.port", "8501",
     "--server.headless", "false"]
)

print("✅ Streamlit running.")
print()

def shutdown(sig, frame):
    print("\n🛑 Shutting down servers...")
    api_proc.terminate()
    st_proc.terminate()
    sys.exit(0)

signal.signal(signal.SIGINT, shutdown)
signal.signal(signal.SIGTERM, shutdown)

# Keep alive
api_proc.wait()
st_proc.wait()
```

**Scoring check:**
- [ ] `subprocess.Popen` used for both servers (not `subprocess.run` which blocks)
- [ ] `time.sleep(2)` between starting FastAPI and Streamlit
- [ ] `api_proc.poll()` checks if FastAPI started successfully
- [ ] `signal.signal(signal.SIGINT, shutdown)` handles Ctrl+C gracefully


---
## 📊 Section 4 — Scoring Rubric

| # | Task | Points | What Claude checks |
|---|------|--------|--------------------|
| 0 | FastAPI: `/projects/platform-summary` endpoint | 10 | 4 platforms returned, correct counts (180/136/132/52), `avg_hourly_rate` rounded to 2dp |
| 1 | `safe_api_call()` + config constants | 10 | Handles 200/401/422/ConnectionError/Timeout, returns `(data, error)` tuple |
| 2 | KPI Dashboard — metrics + platform charts | 20 | 4 `st.metric` values from API (not hardcoded), NRA chart titles, `ttl=60` cache, `st.stop()` on error |
| 3 | Filter page — selectbox → API → table + NRA | 15 | `params={"status":...}` correctly passed, 3 metrics computed from returned df, NRA insight uses `idxmax()` |
| 4 | Predict page — form → API → result display | 15 | Form uses `st.form`, payload keys match Pydantic schema, `st.success`/`st.warning` conditional, `prob:.1%` format |
| 5 | Sidebar: health check + docs links + cache | 10 | `GET /` called for health, offline message shows `uvicorn` command, cache clear uses `st.cache_data.clear()` + `st.rerun()` |
| ★ | `run_stack.py` — launch both servers | 10 | Two `Popen` calls, `time.sleep(2)`, poll check, SIGINT handler |
| **Total** | | **80 + 10★** | |

---

### Deduction Guide

| Error | Deduction |
|---|---|
| API values hardcoded instead of read from response | −5 per instance |
| Chart title doesn't embed actual numbers from data | −3 per chart |
| Missing `@st.cache_data(ttl=60)` on API calls | −3 per call |
| `safe_api_call` missing one error type (e.g. no Timeout handling) | −2 per missing case |
| Form not using `st.form` (bare button instead) | −5 |
| NRA insight uses hardcoded strings instead of computed values | −4 |
| `payload` key names don't match Pydantic model | −5 (predict won't work) |
| `run_stack.py` uses `subprocess.run` (blocks) instead of `Popen` | −5 |


---
## 🎤 Section 5 — Interview / Client Framing

**Q: "Why use FastAPI + Streamlit together instead of just one of them?"**

> "FastAPI and Streamlit solve different problems.  
> FastAPI is a backend API framework — it handles data persistence, authentication,  
> ML model serving, and can serve multiple frontends simultaneously.  
> Streamlit is a frontend framework — great for building interactive dashboards  
> quickly in pure Python.  
> Combining them gives you separation of concerns: the business logic and data  
> live in FastAPI (versioned, testable, reusable), while the UI in Streamlit  
> can be rebuilt or swapped out without touching the API.  
> In a client engagement, I'd typically expose the FastAPI for their internal  
> systems to call programmatically, while the Streamlit dashboard gives  
> non-technical stakeholders a visual interface to the same data."

---

**Q: "How do you handle API authentication in a client-facing dashboard?"**

> "For local development I use a hardcoded API key in a config constant.  
> For production — especially on Streamlit Cloud — I move secrets to  
> `st.secrets`, which reads from a `.streamlit/secrets.toml` file that's  
> excluded from version control via `.gitignore`.  
> The secrets.toml file on Streamlit Cloud is injected at deploy time,  
> so the key never appears in the source code.  
> For higher-security clients, I'd integrate OAuth2 or JWT tokens via  
> FastAPI's OAuth2PasswordBearer — that's a Day 150+ topic."

---

**Q: "What's the difference between `@st.cache_data` and `@st.cache_resource`?"**

> "`st.cache_data` caches the *return value* of a function — data like DataFrames,  
> dicts, lists. It serialises and stores a copy, so each call gets a fresh copy  
> to mutate safely. TTL controls how long the cache lives.  
> `st.cache_resource` caches *global resources* — database connections, ML models,  
> HTTP clients — objects that are expensive to create and should be shared across  
> all users. You wouldn't use cache_resource for API responses because you'd want  
> the data to refresh; you would use it for the trained scikit-learn model so it  
> loads from disk only once per app lifecycle."

---

## 🏁 Day 149 Summary

| Concept | What you built |
|---|---|
| Full-stack architecture | Streamlit → HTTP → FastAPI → SQLite |
| `requests` library | `safe_api_call()` with full error handling |
| API key auth in UI | Header injection, offline detection |
| `st.cache_data(ttl=60)` | API call caching to prevent hammering |
| NRA insights from live data | `idxmax()` / `max()` on API-returned DataFrames |
| `run_stack.py` | `subprocess.Popen` + SIGINT handler |
| Production habit | Never hardcode API values in UI — always read from response |

> **Next up — Day 150:** Docker basics · `Dockerfile` for FastAPI ·  
> `docker build` + `docker run` · docker-compose for the full stack ·  
> Why containers matter for client deployment.
